# Riesgo de crédito en un banco
## Notebook 01 · Exploración de las tablas y sus relaciones

**Entrada:** los 8 CSV de `data/raw/`, descargados en `00_descarga.ipynb`.

**Qué hace este notebook:** revisa cada tabla, comprueba sus claves primarias y foráneas, dibuja el diagrama de relaciones y traduce los códigos en checo.

**Contenido**
1. Carga de las tablas
2. Vistazo general
3. Claves primarias: ¿cada id es único?
4. Claves foráneas: ¿cada referencia existe?
5. Diagrama entidad-relación
6. Traducción de los códigos en checo
7. Variable objetivo: préstamos buenos y malos
8. La trampa del JOIN: préstamos y clientes
9. Rango de fechas

In [1]:
import pandas as pd

nombres = ["account", "card", "client", "disp", "district", "loan", "order", "trans"]

tablas = {
    nombre: pd.read_csv(f"../data/raw/{nombre}.csv", low_memory=False)
    for nombre in nombres
}

for nombre, df in tablas.items():
    print(f"{nombre:9} {df.shape[0]:>10,} filas  {df.shape[1]:>3} columnas")

account        4,500 filas    4 columnas
card             892 filas    4 columnas
client         5,369 filas    4 columnas
disp           5,369 filas    4 columnas
district          77 filas   16 columnas
loan             682 filas    7 columnas
order          6,471 filas    6 columnas
trans      1,056,320 filas   10 columnas


In [2]:
for nombre, df in tablas.items():
    print(f"\n===== {nombre} =====")
    print(df.dtypes.to_string())
    display(df.head(3))


===== account =====
account_id     int64
district_id    int64
frequency        str
date             str


,account_id,district_id,frequency,date
0,1,18,POPLATEK MESICNE,1995-03-24
1,2,1,POPLATEK MESICNE,1993-02-26
2,3,5,POPLATEK MESICNE,1997-07-07



===== card =====
card_id    int64
disp_id    int64
type         str
issued       str


,card_id,disp_id,type,issued
0,1,9,gold,1998-10-16
1,2,19,classic,1998-03-13
2,3,41,gold,1995-09-03



===== client =====
client_id      int64
gender           str
birth_date       str
district_id    int64


,client_id,gender,birth_date,district_id
0,1,F,1970-12-13,18
1,2,M,1945-02-04,1
2,3,F,1940-10-09,1


... (6 outputs left out)


===== order =====
order_id        int64
account_id      int64
bank_to           str
account_to      int64
amount        float64
k_symbol          str


,order_id,account_id,bank_to,account_to,amount,k_symbol
0,29401,1,YZ,87144583,2452.0,SIPO
1,29402,2,ST,89597016,3372.7,UVER
2,29403,2,QR,13943797,7266.0,SIPO



===== trans =====
trans_id        int64
account_id      int64
date              str
type              str
operation         str
amount          int64
balance         int64
k_symbol          str
bank              str
account       float64


,trans_id,account_id,date,type,operation,amount,balance,k_symbol,bank,account
0,1,1,1995-03-24,PRIJEM,VKLAD,1000,1000,NaN,NaN,NaN
1,5,1,1995-04-13,PRIJEM,PREVOD Z UCTU,3679,4679,NaN,AB,41403269.0
2,6,1,1995-05-13,PRIJEM,PREVOD Z UCTU,3679,20977,NaN,AB,41403269.0


---
## 3. Claves primarias

Una **clave primaria** identifica cada fila de forma única: no puede repetirse ni estar vacía. Si un id estuviera repetido, cualquier `JOIN` con esa tabla duplicaría filas.

In [3]:
claves_primarias = {
    "account": "account_id", "card": "card_id", "client": "client_id",
    "disp": "disp_id", "district": "district_id", "loan": "loan_id",
    "order": "order_id", "trans": "trans_id",
}

for tabla, clave in claves_primarias.items():
    df = tablas[tabla]
    unica = df[clave].is_unique
    sin_vacios = df[clave].notna().all()
    print(f"{tabla:9} {clave:12} única: {unica}  sin vacíos: {sin_vacios}")

account   account_id   única: True  sin vacíos: True
card      card_id      única: True  sin vacíos: True
client    client_id    única: True  sin vacíos: True
disp      disp_id      única: True  sin vacíos: True
district  district_id  única: True  sin vacíos: True
loan      loan_id      única: True  sin vacíos: True
order     order_id     única: True  sin vacíos: True
trans     trans_id     única: True  sin vacíos: True


---
## 4. Claves foráneas

Una **clave foránea** es una columna que apunta a la clave primaria de otra tabla (por ejemplo, `loan.account_id` apunta a `account.account_id`). Se comprueba que cada referencia exista: un préstamo de una cuenta inexistente sería un error de integridad.

In [4]:
# (tabla hija, columna) → (tabla padre, columna)
relaciones = [
    ("account", "district_id", "district", "district_id"),
    ("client",  "district_id", "district", "district_id"),
    ("disp",    "client_id",   "client",   "client_id"),
    ("disp",    "account_id",  "account",  "account_id"),
    ("card",    "disp_id",     "disp",     "disp_id"),
    ("loan",    "account_id",  "account",  "account_id"),
    ("order",   "account_id",  "account",  "account_id"),
    ("trans",   "account_id",  "account",  "account_id"),
]

for hija, col_hija, padre, col_padre in relaciones:
    huerfanos = ~tablas[hija][col_hija].isin(tablas[padre][col_padre])
    print(f"{hija}.{col_hija} → {padre}.{col_padre}: {huerfanos.sum()} sin pareja")

account.district_id → district.district_id: 0 sin pareja
client.district_id → district.district_id: 0 sin pareja
disp.client_id → client.client_id: 0 sin pareja
disp.account_id → account.account_id: 0 sin pareja
card.disp_id → disp.disp_id: 0 sin pareja
loan.account_id → account.account_id: 0 sin pareja
order.account_id → account.account_id: 0 sin pareja
trans.account_id → account.account_id: 0 sin pareja


In [5]:
prestamos_por_cuenta = tablas["loan"]["account_id"].value_counts()
print("Cuentas con préstamo:", len(prestamos_por_cuenta))
print("Máximo de préstamos en una cuenta:", prestamos_por_cuenta.max())

Cuentas con préstamo: 682
Máximo de préstamos en una cuenta: 1


---
## 5. Diagrama entidad-relación

```mermaid
erDiagram
    district ||--o{ account : "sucursal de la cuenta"
    district ||--o{ client  : "distrito del cliente"
    client   ||--o{ disp    : "tiene derechos sobre"
    account  ||--o{ disp    : "es usada por"
    disp     ||--o{ card    : "recibe"
    account  ||--o| loan    : "puede tener un"
    account  ||--o{ order   : "tiene"
    account  ||--o{ trans   : "registra"
```

- **`account` es el centro del modelo:** los préstamos, las órdenes de pago y las transacciones cuelgan de la cuenta, no del cliente.
- **`disp` une clientes y cuentas.** Una cuenta puede tener varios clientes (un titular y un usuario autorizado), por eso no hay una columna `client_id` en `account`.
- **El préstamo es de la cuenta.** Para saber de qué cliente es, hay que pasar por `disp`.

---
## 6. Traducción de los códigos en checo

Varias columnas de texto están en checo. Se traducen con diccionarios basados en la descripción original del dataset (PKDD'99). Después de traducir, se comprueba que no haya quedado ningún valor sin traducción.

In [6]:
traducciones = {
    ("account", "frequency"): {
        "POPLATEK MESICNE": "Extracto mensual",
        "POPLATEK TYDNE": "Extracto semanal",
        "POPLATEK PO OBRATU": "Extracto tras cada transacción",
    },
    ("disp", "type"): {
        "OWNER": "Titular",
        "DISPONENT": "Usuario autorizado",
    },
    ("trans", "type"): {
        "PRIJEM": "Ingreso",
        "VYDAJ": "Egreso",
        "VYBER": "Retiro",
    },
    ("trans", "operation"): {
        "VYBER KARTOU": "Retiro con tarjeta",
        "VKLAD": "Depósito en efectivo",
        "PREVOD Z UCTU": "Transferencia recibida",
        "VYBER": "Retiro en efectivo",
        "PREVOD NA UCET": "Transferencia enviada",
    },
    ("trans", "k_symbol"): {
        "POJISTNE": "Seguro",
        "SLUZBY": "Cobro por servicios",
        "UROK": "Intereses ganados",
        "SANKC. UROK": "Interés sancionatorio",
        "SIPO": "Pago del hogar",
        "DUCHOD": "Pensión",
        "UVER": "Cuota de préstamo",
    },
    ("order", "k_symbol"): {
        "POJISTNE": "Seguro",
        "SIPO": "Pago del hogar",
        "LEASING": "Leasing",
        "UVER": "Cuota de préstamo",
    },
}

for (tabla, columna), diccionario in traducciones.items():
    serie = tablas[tabla][columna].astype("string").str.strip()
    sin_traducir = set(serie.dropna().unique()) - set(diccionario) - {""}
    tablas[tabla][columna] = serie.map(diccionario)
    print(f"{tabla}.{columna}: sin traducir → {sin_traducir or 'ninguno'}")

account.frequency: sin traducir → ninguno
disp.type: sin traducir → ninguno
trans.type: sin traducir → ninguno
trans.operation: sin traducir → ninguno
trans.k_symbol: sin traducir → ninguno
order.k_symbol: sin traducir → ninguno


**Dato importante para el modelo:** el código `SANKC. UROK` (**interés sancionatorio**) se cobra cuando el saldo de la cuenta queda en negativo. Es una señal de problemas financieros y probablemente estará muy relacionada con la mora. Pero solo se podrá usar si ocurrió **antes** de la fecha del préstamo: si no, el modelo "vería el futuro".

---
## 7. Variable objetivo: préstamos buenos y malos

La columna `loan.status` tiene cuatro valores:

| Código | Significado | Clasificación |
|---|---|---|
| A | Contrato terminado, sin problemas | Bueno |
| B | Contrato terminado, préstamo no pagado | **Malo** |
| C | Contrato vigente, al día | Bueno |
| D | Contrato vigente, cliente en deuda | **Malo** |

In [7]:
loan = tablas["loan"]

estado = loan["status"].value_counts().sort_index()
display(pd.DataFrame({"préstamos": estado, "%": (estado / len(loan) * 100).round(1)}))

loan["malo"] = loan["status"].isin(["B", "D"]).astype(int)
print(f"Préstamos malos: {loan['malo'].sum()} de {len(loan)} ({loan['malo'].mean():.1%})")

,préstamos,%
status,,
A,203,29.8
B,31,4.5
C,403,59.1
D,45,6.6


Préstamos malos: 76 de 682 (11.1%)


**Decisión:** un préstamo es **malo** si su estado es B o D, y bueno si es A o C. Así se usan los 682 préstamos, con 76 malos (11.1%).

**Limitación:** los préstamos C y D aún están vigentes. Un préstamo C que hoy está al día podría caer en mora más adelante. La alternativa sería usar solo los préstamos terminados (A y B), pero quedarían muy pocos casos. Se acepta esta limitación y se documenta.

**Datos desbalanceados:** solo 1 de cada 9 préstamos es malo. Un modelo que dijera siempre "bueno" acertaría el 88.9% de las veces sin servir para nada. Por eso la exactitud (*accuracy*) no será la métrica principal.

---
## 8. La trampa del JOIN: préstamos y clientes

Para saber de qué cliente es cada préstamo, hay que pasar por `disp`: `loan → disp → client`. Pero una cuenta puede tener **dos clientes** (titular y usuario autorizado).

In [8]:
disp = tablas["disp"]

# JOIN sin filtrar
sin_filtro = loan.merge(disp, on="account_id")
print("Préstamos:", len(loan))
print("Filas después del JOIN sin filtrar:", len(sin_filtro))

# JOIN solo con el titular
titulares = disp[disp["type"] == "Titular"]
con_filtro = loan.merge(titulares, on="account_id")
print("Filas después del JOIN solo con titulares:", len(con_filtro))

assert len(con_filtro) == len(loan), "El JOIN duplicó o perdió préstamos"

Préstamos: 682
Filas después del JOIN sin filtrar: 827
Filas después del JOIN solo con titulares: 682


**Regla para todo el proyecto:** después de cada `JOIN`, se comprueba que el número de filas sea el esperado. Para unir préstamos con clientes se usa solo el **titular** de la cuenta (`disp.type = 'Titular'`).

## 9. Rango de Fechas:

In [9]:
for tabla in ["account", "loan", "trans"]:
    tablas[tabla]["date"] = pd.to_datetime(tablas[tabla]["date"])
    fechas = tablas[tabla]["date"]
    print(f"{tabla:8} {fechas.min().date()} → {fechas.max().date()}")

account  1993-01-01 → 1997-12-29
loan     1993-07-05 → 1998-12-08
trans    1993-01-01 → 1998-12-31


---
## 10. Exportación a SQLite

Las 8 tablas, con los códigos traducidos, la variable `malo` y las fechas en formato `AAAA-MM-DD`, se guardan en una base de datos SQLite para construir las variables con SQL en el siguiente notebook.

In [10]:
import sqlite3

conexion = sqlite3.connect("../data/banco.db")

for nombre, df in tablas.items():
    df = df.copy()
    if "date" in df.columns:
        df["date"] = df["date"].dt.strftime("%Y-%m-%d")
    df.to_sql(nombre, conexion, if_exists="replace", index=False)

conexion.close()
print("Base de datos creada: data/banco.db")

Base de datos creada: data/banco.db
